# Lab Ngày 26: Hạ Tầng MCP/A2A & Agentic Routing

**Khóa học:** AICB-P2T2 · Tuần 6 · Chương 6  
**Framework:** [Google Agent Development Kit (ADK)](https://google.github.io/adk-docs/)  
**Thời lượng:** ~2 giờ

---

## Câu hỏi mở đầu

> *"Agent gọi agent — hạ tầng của bạn có scale được cho multi-agent choreography không?"*

Một hệ thống nghiên cứu cần 4 agent chuyên biệt phối hợp. Nếu không có chuẩn giao tiếp chung, mỗi agent nói "ngôn ngữ" riêng. **MCP + A2A** là giao thức phổ quát cho hệ sinh thái AI agent.

---

## Mục tiêu học tập

Sau buổi lab này, bạn sẽ:

1. Thiết kế và triển khai **hạ tầng MCP server** cho AI tools
2. Triển khai **giao tiếp A2A** giữa nhiều agent bằng ADK
3. Xây dựng **lớp agentic routing** với semantic routing
4. Áp dụng các mẫu **state, bảo mật và observability** trong hệ multi-agent

## Sản phẩm cuối ngày

| Thành phần | Yêu cầu |
|------------|--------|
| MCP server | 3 tools qua stdio (local) hoặc SSE (remote) |
| Agent registry | Health check + khám phá capability |
| Semantic router | Định tuyến request tới đúng specialist |
| Demo multi-agent | Orchestrator + 2 specialist qua A2A |
| Trace | Toàn bộ luồng multi-agent hiển thị trong log/trace |
| Data governance | Policy MCP/A2A + audit log + HITL gate |

---
## Module 0 — Cài đặt môi trường

Lab dùng **Conda** (môi trường khuyến nghị: `pii-env`). Trước khi chạy cell bên dưới:

```bash
conda create -n pii-env python=3.12 -y   # chỉ lần đầu
conda activate pii-env
```

Chọn kernel **pii-env** trong Jupyter nếu notebook hỏi.

Cài ADK kèm **extra A2A** và MCP SDK. Gói `google-adk` cơ bản **không** bao gồm dependency A2A.

In [21]:
# Chạy một lần sau khi conda activate pii-env (kernel Jupyter = pii-env)
!pip install -q -r requirements.txt

# Nếu vẫn thấy cảnh báo cryptography từ presidio cũ (2.2.360), nâng cấp:
# !pip install -q -U "presidio-anonymizer>=2.2.363"

In [22]:
import os
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

# Google AI Studio (khuyến nghị cho lab)
os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "FALSE")

assert os.getenv("GOOGLE_API_KEY"), (
    "Đặt GOOGLE_API_KEY trong .env — lấy key tại https://aistudio.google.com/app/apikey"
)

print("✓ Môi trường sẵn sàng")
print(f"  Thư mục dự án: {PROJECT_ROOT}")

✓ Môi trường sẵn sàng
  Thư mục dự án: e:\Programming\AI in action assginments\Day26-2A202600857-Pham-Tran-Nguyen-Phu


### Tổng quan kiến trúc

```
┌─────────────────────────────────────────────────────────────┐
│                     ORCHESTRATOR (ADK)                       │
│  Semantic Router → ủy quyền cho các specialist               │
└──────────┬──────────────────────────────┬───────────────────┘
           │ A2A (HTTP)                   │ MCP (stdio/SSE)
           ▼                              ▼
┌──────────────────┐            ┌──────────────────────┐
│  Search Agent    │            │  MCP Tools Server     │
│  :8011           │            │  search / sql / sum   │
├──────────────────┤            └──────────────────────┘
│  Database Agent  │
│  :8012           │
└──────────────────┘
```

**Điểm then chốt:** MCP chuẩn hóa truy cập *tool*; A2A chuẩn hóa giao tiếp *agent*. ADK kết nối cả hai.

---
## ⚠️ Bước bắt buộc — Khởi động A2A Specialist Servers

**Phải làm trước Module 2, Module 4 và Capstone.** Nếu bỏ qua, kiểm tra agent card sẽ báo `[CHƯA ĐẠT]`.

**Điều kiện:** đã cài dependency (Module 0) và **đang ở thư mục dự án**. Kích hoạt conda env:

```bash
conda activate pii-env
export PYTHONPATH=$PWD
```

### Cách 1 — Từ notebook (khuyến nghị)

Chạy cell bên dưới để khởi động cả hai server nền. Đợi thông báo `search OK` / `database OK`, rồi chạy cell kiểm tra.

### Cách 2 — Hai terminal riêng

```bash
cd Day26-MCP_A2A_Infrastructure
conda activate pii-env
export PYTHONPATH=$PWD

# Terminal 1
python -m uvicorn agents.search_agent.agent:a2a_app --host localhost --port 8011

# Terminal 2
python -m uvicorn agents.database_agent.agent:a2a_app --host localhost --port 8012
```

Hoặc: `bash scripts/start_search_agent.sh` và `bash scripts/start_database_agent.sh`

### Cách 3 — Một lệnh (nền)

```bash
bash scripts/start_a2a_servers.sh
```

### Kiểm tra / Dừng

```bash
curl http://localhost:8011/.well-known/agent-card.json
curl http://localhost:8012/.well-known/agent-card.json
bash scripts/stop_a2a_servers.sh
```

In [23]:
# Khoi dong A2A servers nen (chay cell nay mot lan moi phien lab)
# Dung truc tiep uvicorn qua Python (khong phu thuoc bash/lsof/nohup) de chay duoc tren Windows lan Mac/Linux.
import subprocess
import sys
import time

import httpx

LOGS_DIR = PROJECT_ROOT / "logs"
LOGS_DIR.mkdir(exist_ok=True)

AGENTS_TO_START = [("search_agent", 8011), ("database_agent", 8012), ("synthesis_agent", 8013)]


def _agent_card_ok(port: int) -> bool:
    try:
        r = httpx.get(f"http://localhost:{port}/.well-known/agent-card.json", timeout=1)
        return r.status_code == 200
    except Exception:
        return False


_a2a_procs = globals().get("_a2a_procs", {})

for name, port in AGENTS_TO_START:
    if _agent_card_ok(port):
        print(f"OK {name} da chay san (agent-card hop le) o cong {port}")
        continue
    log_path = LOGS_DIR / f"{name}.log"
    log_file = open(log_path, "w", encoding="utf-8")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", f"agents.{name}.agent:a2a_app", "--host", "localhost", "--port", str(port)],
        cwd=PROJECT_ROOT,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )
    _a2a_procs[name] = proc
    print(f"-> Da khoi dong {name} (PID {proc.pid}) tren cong {port}, log: {log_path}")

globals()["_a2a_procs"] = _a2a_procs

print("Doi server san sang...")
for _ in range(10):
    time.sleep(1)
    if all(_agent_card_ok(port) for _, port in AGENTS_TO_START):
        print("Tat ca A2A servers da san sang.")
        break
else:
    print("Van chua thay du 3 agent-card hop le - kiem tra logs/<agent>.log, hoac co the cong dang bi service khac chiem (kiem tra bang: Get-NetTCPConnection -LocalPort 8011,8012,8013).")


-> Da khoi dong search_agent (PID 23088) tren cong 8011, log: e:\Programming\AI in action assginments\Day26-2A202600857-Pham-Tran-Nguyen-Phu\logs\search_agent.log
-> Da khoi dong database_agent (PID 37896) tren cong 8012, log: e:\Programming\AI in action assginments\Day26-2A202600857-Pham-Tran-Nguyen-Phu\logs\database_agent.log
-> Da khoi dong synthesis_agent (PID 37120) tren cong 8013, log: e:\Programming\AI in action assginments\Day26-2A202600857-Pham-Tran-Nguyen-Phu\logs\synthesis_agent.log
Doi server san sang...
Tat ca A2A servers da san sang.


In [24]:
# Kiểm tra agent card — phải thấy ✓ trước khi sang Module 4 / Capstone
import httpx

AGENT_CARDS = {
    "search_agent": "http://localhost:8011/.well-known/agent-card.json",
    "database_agent": "http://localhost:8012/.well-known/agent-card.json",
    "synthesis_agent": "http://localhost:8013/.well-known/agent-card.json",
}

all_ok = True
for name, url in AGENT_CARDS.items():
    try:
        r = httpx.get(url, timeout=5)
        r.raise_for_status()
        card = r.json()
        print(f"✓ {name}: {card.get('name')} — {str(card.get('description', ''))[:50]}...")
    except Exception as exc:
        all_ok = False
        print(f"✗ {name}: {exc}")
        port = {"search_agent": 8011, "database_agent": 8012, "synthesis_agent": 8013}[name]
        print(f"  → Chạy: uvicorn agents.{name}.agent:a2a_app --host localhost --port {port}")

if all_ok:
    print("\n✓ A2A servers sẵn sàng — tiếp tục lab")
else:
    print("\n✗ Chưa sẵn sàng — chạy: bash scripts/start_a2a_servers.sh")

✓ search_agent: search_agent — Tìm kiếm web và trả về đoạn trích liên quan cho tá...
✓ database_agent: database_agent — Truy vấn database chỉ đọc và trả về metrics có cấu...
✓ synthesis_agent: synthesis_agent — Tổng hợp kết quả nghiên cứu thành báo cáo cuối có ...

✓ A2A servers sẵn sàng — tiếp tục lab


---
## Module 1 — Model Context Protocol (MCP)

### Lý thuyết (từ slide)

| Lớp | Vai trò |
|-----|--------|
| **Tools Server** | Thực thi hàm (search, SQL, API) |
| **Resources Server** | Expose dữ liệu (file, hàng DB) |
| **Prompts Server** | Template prompt tái sử dụng |

**Transport:** `stdio` (dev local), `HTTP+SSE` (remote/K8s), `WebSocket` (streaming hai chiều)

**Quy tắc thiết kế:**
- Docstring rõ ràng (LLM đọc để chọn tool)
- Type hint cho mọi tham số
- Validate input trước khi thực thi
- SQL tool mặc định chỉ đọc (read-only)
- Log mọi lần gọi tool

### 📝 Bài tập 1.1 — Khám phá MCP Server

Mở `mcp_server/research_tools_server.py` và trả lời:
1. Ba tool nào được expose?
2. `_sql_query` enforce governance như thế nào?
3. Vì sao dùng transport stdio khi dev local?

In [25]:
# Demo: gọi logic MCP tool trực tiếp (không cần khởi động server)
import sys
sys.path.insert(0, str(PROJECT_ROOT / "mcp_server"))

from research_tools_server import _search_documents, _sql_query, _summarize_text

print("=== search_documents ===")
print(_search_documents("MCP"))

print("\n=== sql_query (cho phép) ===")
print(_sql_query("SELECT * FROM agent_metrics"))

print("\n=== sql_query (bị chặn) ===")
try:
    _sql_query("DROP TABLE agent_metrics")
except ValueError as exc:
    print(f"Đã chặn: {exc}")

print("\n=== summarize_text ===")
print("\n".join(_summarize_text("MCP build một lần. A2A kết nối agent. Routing chọn specialist.")))

=== search_documents ===
[{'id': 'doc-1', 'title': 'VinUni AI Curriculum Overview', 'body': 'Phase 2 Track 2 covers MCP, A2A, and multi-agent orchestration.'}, {'id': 'doc-2', 'title': 'MCP Transport Options', 'body': 'MCP supports stdio for local dev and HTTP+SSE for remote deployment.'}]

=== sql_query (cho phép) ===
[{'agent': 'search_agent', 'tasks_completed': 42, 'avg_latency_ms': 820}, {'agent': 'database_agent', 'tasks_completed': 31, 'avg_latency_ms': 1100}, {'agent': 'synthesis_agent', 'tasks_completed': 28, 'avg_latency_ms': 2400}]

=== sql_query (bị chặn) ===

=== summarize_text ===
- MCP build một lần
- A2A kết nối agent
- Routing chọn specialist


### Kết nối MCP Tools với ADK Agent

ADK bọc MCP server qua `McpToolset`. Agent orchestrator trong repo đã bind cả 3 tool.

In [26]:
from google.adk.tools.mcp_tool import StdioConnectionParams
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from mcp import StdioServerParameters

mcp_server_path = PROJECT_ROOT / "mcp_server" / "research_tools_server.py"

mcp_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="python",
            args=[str(mcp_server_path)],
        ),
        timeout=10,
    ),
    tool_filter=["search_documents", "sql_query", "summarize_text"],
)

print("✓ Đã cấu hình McpToolset")
print(f"  Server: {mcp_server_path.name}")
print(f"  Bộ lọc: search_documents, sql_query, summarize_text")

✓ Đã cấu hình McpToolset
  Server: research_tools_server.py
  Bộ lọc: search_documents, sql_query, summarize_text


### 📝 Bài tập 1.2 — Thêm tool MCP thứ tư

**Nhiệm vụ:** Mở rộng `research_tools_server.py` với tool `count_words` trả về số từ trong chuỗi.

Checklist:
- [ ] Thêm vào `list_tools()` với `inputSchema` đúng
- [ ] Triển khai trong `call_tool()`
- [ ] Thêm vào `tool_filter` trong orchestrator agent
- [ ] Test qua ADK agent hoặc gọi hàm trực tiếp

---
## Module 2 — Giao thức Agent-to-Agent (A2A)

### Lý thuyết

A2A coi agent như **microservices**:

| Khái niệm | Mô tả |
|-----------|-------|
| **Agent Card** | JSON tại `/.well-known/agent-card.json` — capability & schema |
| **Task** | Đơn vị công việc với các trạng thái vòng đời |
| **Message** | Giao tiếp trong một task |

### Vòng đời Task

```
Submitted → Working → Input Required → Completed
                  ↘ Failed / Canceled
```

Task có thể **tạm dừng** ở `input-required` — agent yêu cầu caller cung cấp thêm thông tin trước khi tiếp tục.

### Ánh xạ sang ADK

| Hành động | API ADK |
|----------|--------|
| Expose agent làm A2A server | `to_a2a(agent, port=8011)` |
| Tiêu thụ agent remote | `RemoteA2aAgent(name, description, agent_card=url)` |
| Thay thế qua CLI | `adk api_server --a2a --port 8011 agents/search_agent` |

### 🚀 Thực hành — Kiểm tra lại A2A (Module 2)

> **Đã khởi động ở Module 0.5 chưa?** Nếu chưa, quay lại phầ **「Bước bắt buộc — Khởi động A2A Specialist Servers」** và chạy cell khởi động + kiểm tra.

Cell bên dưới lặp lại bước xác nhận agent card (giống Module 0.5).

In [27]:
import httpx

AGENT_CARDS = {
    "search": "http://localhost:8011/.well-known/agent-card.json",
    "database": "http://localhost:8012/.well-known/agent-card.json",
}

for label, url in AGENT_CARDS.items():
    try:
        response = httpx.get(url, timeout=3)
        response.raise_for_status()
        card = response.json()
        print(f"✓ {label}_agent: {card.get('name')} — {card.get('description', '')[:60]}...")
    except Exception as exc:
        print(f"✗ Không kết nối được {label}_agent: {exc}")
        print(f"  Khởi động: uvicorn agents.{label}_agent.agent:a2a_app --port {8011 if label == 'search' else 8012}")

✓ search_agent: search_agent — Tìm kiếm web và trả về đoạn trích liên quan cho tác vụ nghiê...
✓ database_agent: database_agent — Truy vấn database chỉ đọc và trả về metrics có cấu trúc....


### Tiêu thụ Remote Agent từ Orchestrator

Orchestrator dùng `RemoteA2aAgent` làm sub-agent. LLM coordinator của ADK ủy quyền dựa trên `description` của từng sub-agent.

In [28]:
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent

remote_search = RemoteA2aAgent(
    name="search_agent",
    description="Tìm kiếm tài liệu và web để thu thập bằng chứng nghiên cứu.",
    agent_card=AGENT_CARDS["search"],
)

remote_database = RemoteA2aAgent(
    name="database_agent",
    description="Chạy SQL chỉ đọc trên bảng metrics của agent.",
    agent_card=AGENT_CARDS["database"],
)

print("✓ Đã cấu hình proxy RemoteA2aAgent")
print(f"  search → {AGENT_CARDS['search']}")
print(f"  database → {AGENT_CARDS['database']}")

✓ Đã cấu hình proxy RemoteA2aAgent
  search → http://localhost:8011/.well-known/agent-card.json
  database → http://localhost:8012/.well-known/agent-card.json


### 📝 Bài tập 2.1 — A2A vs Sub-Agent Local

Điền bảng theo hiểu biết của bạn:

| Tiêu chí | A2A (Remote) | Sub-Agent Local |
|----------|-------------|------------------|
| Triển khai | Agent chạy như một service độc lập (process/server riêng, có thể khác máy, khác ngôn ngữ), giao tiếp qua HTTP + Agent Card JSON (`to_a2a()`, `RemoteA2aAgent`) | Agent chạy trong cùng tiến trình Python, khai báo trực tiếp trong `sub_agents=[...]` của agent cha, không cần network |
| Hiệu năng | Có overhead mạng: serialize/deserialize JSON, HTTP round-trip, timeout — độ trễ cao hơn | Gọi hàm/method trực tiếp trong cùng tiến trình — gần như không overhead, độ trễ thấp nhất |
| Cô lập state | Mỗi agent có state, session và vòng đời riêng biệt; agent này crash không kéo sập agent khác; dễ scale/deploy độc lập | Chia sẻ chung process và bộ nhớ với agent cha; state dễ rò rỉ qua context nếu không cẩn thận; một agent lỗi có thể ảnh hưởng cả hệ |
| Phù hợp khi | - Agent do team/tổ chức khác phát triển và vận hành<br>- Cần scale độc lập theo tải (VD: search_agent cần nhiều instance hơn database_agent)<br>- Cần polyglot (agent viết bằng ngôn ngữ/framework khác)<br>- Cần ranh giới governance/audit rõ ràng theo từng agent | - Tất cả agent cùng team, cùng codebase, cùng vòng đời triển khai<br>- Ưu tiên độ trễ thấp, gọi liên tục trong một tác vụ<br>- Không cần scale riêng lẻ từng agent<br>- Prototype nhanh, không muốn vận hành nhiều service |

**Thảo luận:** Khi nào chọn A2A thay vì `sub_agents=[local_agent]` trong ADK?

Chọn **A2A** khi agent cần được đối xử như một microservice độc lập: có ranh giới bảo mật/governance riêng (như capability matrix trong lab — mỗi agent chỉ được gọi bởi những caller nhất định), cần scale hoặc triển khai độc lập với orchestrator, hoặc agent nằm ngoài quyền kiểm soát trực tiếp của mình (team khác, bên thứ ba, ngôn ngữ khác). Đổi lại phải chấp nhận độ trễ mạng và độ phức tạp vận hành (phải khởi động, theo dõi, và đảm bảo từng server luôn sẵn sàng).

Chọn **sub-agent local** khi tất cả agent thuộc cùng một đội/codebase, ưu tiên độ trễ thấp và đơn giản hoá vận hành (không cần chạy nhiều server, không cần agent card, không có network là điểm lỗi tiềm ẩn) — phù hợp cho các bước xử lý nội bộ, gọi thường xuyên trong một luồng duy nhất.


---
## Module 3 — Mẫu Agentic Routing

### So sánh chiến lược routing

| Chiến lược | Ưu điểm | Nhược điểm | Khi nào dùng |
|------------|---------|------------|-------------|
| **Keyword** | Nhanh, đơn giản | Dễ vỡ, bỏ sót ngữ cảnh | ≤5 agent |
| **Embedding** | Bền, scale tốt | Chậm hơn, cần embedding | 5–50 agent |
| **LLM-based** | Linh hoạt, hiểu ngữ cảnh | Đắt, chậm | Routing phức tạp |

**Semantic routing:** embed request → cosine similarity với mô tả capability của agent → top-k ứng viên.

**Fallback chain:** agent chính → agent dự phòng → leo thang lên người — không để user bị kẹt.

In [29]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from lab_utils.semantic_router import AgentCapability, SemanticRouter

router = SemanticRouter(
    agents=[
        AgentCapability(
            name="search_agent",
            description="Tìm kiếm web tài liệu nghiên cứu bằng chứng",
            tags=["search", "web", "documents"],
        ),
        AgentCapability(
            name="database_agent",
            description="SQL metrics phân tích database truy vấn SELECT",
            tags=["sql", "metrics", "database"],
        ),
        AgentCapability(
            name="synthesis_agent",
            description="Tóm tắt kết hợp kết quả thành báo cáo cuối",
            tags=["summary", "report", "synthesis"],
        ),
    ]
)

test_queries = [
    "Tìm bài viết về việc áp dụng giao thức MCP",
    "SELECT độ trễ trung bình từ agent_metrics",
    "Viết tóm tắt một trang về kết quả nghiên cứu",
    "Xin chào, bạn làm được gì?",
]

print(f"{'Truy vấn':<50} {'Định tuyến tới':<20} Điểm")
print("-" * 80)
for query in test_queries:
    candidates = router.route(query, top_k=1)
    target = router.route_with_fallback(query, fallback="orchestrator")
    score = candidates[0][1] if candidates else 0.0
    print(f"{query[:48]:<50} {target:<20} {score:.3f}")

Truy vấn                                           Định tuyến tới       Điểm
--------------------------------------------------------------------------------
Tìm bài viết về việc áp dụng giao thức MCP         synthesis_agent      0.534
SELECT độ trễ trung bình từ agent_metrics          synthesis_agent      0.365
Viết tóm tắt một trang về kết quả nghiên cứu       synthesis_agent      0.758
Xin chào, bạn làm được gì?                         search_agent         0.365


### Mẫu Agent Registry

Trên production, agent đăng ký khi khởi động và hủy đăng ký khi tắt. Registry lưu metadata capability để discovery.

In [30]:
from lab_utils.agent_registry import AgentRegistry, RegisteredAgent

registry = AgentRegistry()

registry.register(
    RegisteredAgent(
        name="search_agent",
        url="http://localhost:8011",
        description="Chuyên gia tìm kiếm web và tài liệu",
        capabilities={"skills": ["search_web"], "transport": "a2a"},
    )
)
registry.register(
    RegisteredAgent(
        name="database_agent",
        url="http://localhost:8012",
        description="Chuyên gia SQL metrics chỉ đọc",
        capabilities={"skills": ["run_sql_query"], "transport": "a2a"},
    )
)

print("Các agent đã đăng ký:")
for agent in registry.list_agents():
    status = "khỏe mạnh" if agent.healthy else "không khỏe"
    print(f"  • {agent.name} ({status}) @ {agent.url}")

print("\nTìm capability 'sql':")
for agent in registry.find_by_capability("sql"):
    print(f"  → {agent.name}")

Các agent đã đăng ký:
  • search_agent (khỏe mạnh) @ http://localhost:8011
  • database_agent (khỏe mạnh) @ http://localhost:8012

Tìm capability 'sql':
  → database_agent


### 📝 Bài tập 3.1 — Xây dựng Fallback Chain

**Nhiệm vụ:** Mở rộng `SemanticRouter.route_with_fallback` để nhận danh sách fallback có thứ tự:

```python
def route_with_chain(self, request: str, chain: list[str]) -> str:
    """Thử route chính; nếu điểm < ngưỡng, đi theo chuỗi fallback."""
    ...
```

Test với: `chain=["search_agent", "database_agent", "orchestrator"]`

In [31]:
# Test route_with_chain — Bài tập 3.1
chain = ["search_agent", "database_agent", "orchestrator"]

test_queries_chain = [
    "Tìm bài viết về việc áp dụng giao thức MCP",
    "SELECT độ trễ trung bình từ agent_metrics",
    "Viết tóm tắt một trang về kết quả nghiên cứu",
    "Xin chào, bạn làm được gì?",
    "abc xyz không liên quan gì cả",
]

print(f"{'Truy vấn':<50} {'route_with_chain':<20}")
print("-" * 75)
for query in test_queries_chain:
    target = router.route_with_chain(query, chain)
    print(f"{query[:48]:<50} {target:<20}")

# Ép ngưỡng cao hơn để minh hoạ các bước fallback giữa chuỗi thực sự được dùng
strict_router = SemanticRouter(agents=router.agents, threshold=0.6)
print()
print("Với threshold=0.6 (buộc đi qua chuỗi fallback nhiều hơn):")
for query in test_queries_chain:
    target = strict_router.route_with_chain(query, chain)
    print(f"{query[:48]:<50} {target:<20}")


Truy vấn                                           route_with_chain    
---------------------------------------------------------------------------
Tìm bài viết về việc áp dụng giao thức MCP         synthesis_agent     
SELECT độ trễ trung bình từ agent_metrics          synthesis_agent     
Viết tóm tắt một trang về kết quả nghiên cứu       synthesis_agent     
Xin chào, bạn làm được gì?                         search_agent        
abc xyz không liên quan gì cả                      search_agent        

Với threshold=0.6 (buộc đi qua chuỗi fallback nhiều hơn):
Tìm bài viết về việc áp dụng giao thức MCP         orchestrator        
SELECT độ trễ trung bình từ agent_metrics          database_agent      
Viết tóm tắt một trang về kết quả nghiên cứu       synthesis_agent     
Xin chào, bạn làm được gì?                         orchestrator        
abc xyz không liên quan gì cả                      orchestrator        


---
## Module 4 — Demo Full Luồng (chạy được)

### Mẫu Orchestrator

```
Yêu cầu người dùng
     │
     ▼
Orchestrator (phân rã + định tuyến)
     ├──► Search Agent (A2A :8011)
     ├──► Database Agent (A2A :8012)
     └──► MCP Tools (stdio: search / sql / summarize)
```

**Điều kiện:** Module 0.5 đã chạy — A2A servers ✓ trên cổng 8011/8012.

Hai ví dụ bên dưới chạy **orchestrator → A2A + MCP → tổng hợp** qua `run_full_flow()`.

In [32]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from lab_utils.full_flow import check_a2a_servers, print_flow_summary, run_full_flow

ok, errors = check_a2a_servers()
if not ok:
    raise RuntimeError(
        "A2A servers chưa sẵn sàng — quay lại Module 0.5.\n" + "\n".join(errors)
    )
print("✓ A2A servers OK — sẵn sàng chạy full luồng")

✓ A2A servers OK — sẵn sàng chạy full luồng


### Ví dụ 1 — A2A: Orchestrator → Search Agent

Luồng: user → orchestrator → **transfer A2A** → `search_agent` → tổng hợp.

In [33]:
# Ví dụ 1 — A2A search delegation
result_1 = await run_full_flow(
    "Transfer sang search_agent để tìm kiếm web về multi-agent orchestration.",
    verbose=True,
)
print_flow_summary(result_1)

Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno
Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno


[orchestrator] Tôi sẽ ủy quyền cho search_agent để tìm kiếm web về multi-agent orchestration.
[search_agent] Multi-agent orchestration là một lĩnh vực mới nổi, và hiện tại tôi chưa tìm được nhiều thông tin chi tiết về nó. Tuy nhiên, tôi đã tìm thấy một số đoạn trích liên quan đến các khái niệm như chuẩn hóa giao diện tool trên các LLM framework (Kết quả cho: multi-agent orchestration) và giao thức A2A để m...

Trace ID : 0f8c05fa-5971-457b-9d26-53b79224c52d
Agents   : orchestrator, search_agent
Sự kiện  : 2 lượt
Multi-agent orchestration là một lĩnh vực mới nổi, và hiện tại tôi chưa tìm được nhiều thông tin chi tiết về nó. Tuy nhiên, tôi đã tìm thấy một số đoạn trích liên quan đến các khái niệm như chuẩn hóa giao diện tool trên các LLM framework (Kết quả cho: multi-agent orchestration) và giao thức A2A để mô tả khả năng của agent (Tổng quan giao thức A2A).

Bạn có muốn tôi tìm kiếm thêm thông tin về các khía cạnh cụ thể hơn của multi-agent orchestration không?


### Ví dụ 2 — MCP + A2A: Tài liệu + Metrics + Tổng hợp

Luồng: user → orchestrator → **MCP** `search_documents` + `sql_query` → tổng hợp báo cáo.

*(Orchestrator cũng có thể ủy quyền `database_agent` qua A2A cho bước SQL — thử đổi prompt nếu muốn.)*

In [34]:
# Ví dụ 2 — MCP multi-tool + tổng hợp
result_2 = await run_full_flow(
    "Bước 1: dùng search_documents tìm MCP. "
    "Bước 2: dùng sql_query SELECT * FROM agent_metrics. "
    "Bước 3: tóm tắt kết quả thành báo cáo ngắn.",
    verbose=True,
)
print_flow_summary(result_2)

Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno
Error on session runner task: fileno
Error on session runner task: fileno
Failed to get tools from toolset McpToolset: Failed to create MCP session: Failed to create MCP session: fileno


[orchestrator] Tôi hiểu bạn muốn thực hiện một chuỗi các bước. Tôi sẽ bắt đầu bằng cách ủy quyền cho search_agent để tìm kiếm tài liệu về MCP.
[search_agent] Tôi đã tìm kiếm MCP và nhận được các kết quả sau:

*   **Kết quả cho: MCP**: "MCP chuẩn hóa giao diện tool trên các LLM framework."
*   **Tổng quan giao thức A2A**: "Agent card tại /.well-known/agent-card.json mô tả capability."

Tôi không thể thực hiện truy vấn SQL "SELECT * FROM agent_metrics" vì ...

Trace ID : 697ed296-e7bf-45ed-bd1d-905882757dfe
Agents   : orchestrator, search_agent
Sự kiện  : 2 lượt
Tôi đã tìm kiếm MCP và nhận được các kết quả sau:

*   **Kết quả cho: MCP**: "MCP chuẩn hóa giao diện tool trên các LLM framework."
*   **Tổng quan giao thức A2A**: "Agent card tại /.well-known/agent-card.json mô tả capability."

Tôi không thể thực hiện truy vấn SQL "SELECT * FROM agent_metrics" vì tôi là tác nhân tìm kiếm và không có khả năng truy vấn cơ sở dữ liệu.

Dưới đây là bản tóm tắt ngắn gọn về kết quả tìm kiếm MCP:

MCP

### Capstone Demo — ADK Web UI (sinh viên tự chạy)

ADK Web là giao diện chat tương tác cho orchestrator. Khác với `run_full_flow()` trong notebook, bạn gõ prompt trực tiếp tại trình duyệt.

#### Kiến trúc khi chạy capstone

```
localhost:8020  ADK Web (orchestrator)
     ├── A2A → search_agent      :8011
     ├── A2A → database_agent    :8012
     ├── A2A → synthesis_agent   :8013
     └── MCP   research_tools    (stdio, spawn tự động)
```

#### Cách khởi động (khuyến nghị — một lệnh)

```bash
cd Day26-MCP_A2A_Infrastructure
conda activate pii-env
export PYTHONPATH=$PWD
bash scripts/start_capstone.sh
```

Script trên sẽ: (1) khởi động 3 A2A specialist nền, (2) mở ADK Web tại http://localhost:8020.

#### Cách khởi động thủ công (2 terminal)

**Terminal 1 — A2A specialists:**
```bash
bash scripts/start_a2a_servers.sh
```

**Terminal 2 — ADK Web:**
```bash
bash scripts/start_adk_web.sh
# hoặc: adk web agents/orchestrator
```

```bash
# ❌ SAI — sẽ báo "No root_agent found for 'agents'"
adk web agents
```

#### Công cụ mới trong capstone

| Thành phần | File | Vai trò |
|------------|------|--------|
| `start_capstone.sh` | `scripts/` | Khởi động A2A + ADK Web một lệnh |
| `synthesis_agent` | `agents/synthesis_agent/` | Specialist tổng hợp báo cáo (:8013) |
| `suggest_routing` | `lab_utils/routing_tool.py` | Gợi ý semantic routing (tư vấn) |
| Auto `trace_id` | `adk_callbacks.py` | Tự sinh trace cho ADK Web (governance A2A) |

Chạy cell **kiểm tra trước khi mở ADK Web** bên dưới, rồi làm bài tập ghi kết quả.

In [35]:
# Kiểm tra trước khi mở ADK Web — chạy cell này trong notebook
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import httpx
from lab_utils.routing_tool import suggest_routing

CAPSTONE_CARDS = {
    "search_agent": "http://localhost:8011/.well-known/agent-card.json",
    "database_agent": "http://localhost:8012/.well-known/agent-card.json",
    "synthesis_agent": "http://localhost:8013/.well-known/agent-card.json",
}

print("=== Kiểm tra A2A specialists ===")
all_ok = True
for name, url in CAPSTONE_CARDS.items():
    try:
        r = httpx.get(url, timeout=5)
        r.raise_for_status()
        print(f"✓ {name}")
    except Exception as exc:
        all_ok = False
        print(f"✗ {name}: {exc}")

print("\n=== Demo suggest_routing (orchestrator tool) ===")
for q in [
    "Tìm bài viết về MCP",
    "SELECT avg latency FROM agent_metrics",
    "Viết báo cáo tổng hợp kết quả nghiên cứu",
]:
    result = suggest_routing(q)
    print(f"  '{q[:40]}...' → {result['recommended_agent']} (top score: {result['top_candidates'][0]['score']})")

if all_ok:
    print("\n✓ Sẵn sàng — mở terminal và chạy: bash scripts/start_capstone.sh")
    print("  Sau đó truy cập http://localhost:8020")
else:
    print("\n✗ Chưa sẵn sàng — chạy: bash scripts/start_a2a_servers.sh")

=== Kiểm tra A2A specialists ===
✓ search_agent
✓ database_agent
✓ synthesis_agent

=== Demo suggest_routing (orchestrator tool) ===
  'Tìm bài viết về MCP...' → synthesis_agent (top score: 0.606)
  'SELECT avg latency FROM agent_metrics...' → database_agent (top score: 0.261)
  'Viết báo cáo tổng hợp kết quả nghiên cứu...' → synthesis_agent (top score: 0.766)

✓ Sẵn sàng — mở terminal và chạy: bash scripts/start_capstone.sh
  Sau đó truy cập http://localhost:8020


### 📝 Bài tập sinh viên — Ghi kết quả ADK Web

**Hướng dẫn:** Sau khi `start_capstone.sh` chạy, mở http://localhost:8020. **Tạo session mới** (nút +), gõ từng prompt bên dưới.

**Đọc kết quả trong ADK Web:**
| Bubble | Ý nghĩa |
|--------|---------|
| `State: trace_id, task_id...` | Governance khởi tạo session — **không phải câu trả lời** |
| Text từ orchestrator | Câu trả lời chính |
| Tab **Trace** (bên phải) | Xem `transfer_to_agent`, MCP calls, A2A events |

Nếu chat trống sau prompt → mở **Trace**, kiểm tra có `transfer_to_agent` không. Restart: `Ctrl+C` rồi `bash scripts/start_capstone.sh`.

| # | Prompt (copy vào ADK Web) | Kỳ vọng |
|---|---------------------------|---------|
| **W1** | `Tôi cần tìm web về multi-agent orchestration. Hãy transfer_to_agent sang search_agent và trả kết quả.` | A2A → search_agent + text |
| **W2** | `Bước 1: dùng search_documents tìm MCP. Bước 2: dùng sql_query SELECT * FROM agent_metrics. Bước 3: tóm tắt báo cáo ngắn.` | MCP: search_documents + sql_query |
| **W3** | `Ủy quyền synthesis_agent tổng hợp báo cáo executive từ các findings về MCP và A2A.` | A2A → synthesis_agent |
| **W4** | `Gọi suggest_routing rồi giải thích bạn sẽ chọn agent nào: "SELECT độ trễ trung bình từ agent_metrics"` | Tool suggest_routing → database_agent |
| **W5** | `DROP TABLE agent_metrics` (thử governance) | Bị chặn — blocked / deny |

**Quan sát thêm (ghi vào bảng):**
- Trace ID có trong session state không?
- Agent nào xuất hiện trong Trace? (`orchestrator`, `search_agent`, …)
- `tail -5 logs/governance_audit.jsonl`

In [ ]:
# 📝 SINH VIÊN ĐIỀN KẾT QUẢ ADK WEB — thay thế placeholder bằng quan sát thực tế

adk_web_results = [
    {
        "prompt_id": "W1",
        "agents_involved": ["orchestrator", "search_agent"],
        "tools_or_protocol": "A2A (transfer_to_agent)",
        "outcome": "ĐẠT",
        "notes": "transfer_to_agent('search_agent') chạy đúng, search_agent trả kết quả về multi-agent orchestration.",
    },
    {
        "prompt_id": "W2",
        "agents_involved": ["orchestrator"],
        "tools_or_protocol": "MCP (search_documents, sql_query, summarize_text)",
        "outcome": "ĐẠT",
        "notes": "Gọi tuần tự đúng 3 bước, sql_query trả agent_metrics thật, summarize_text tổng hợp báo cáo ngắn.",
    },
    {
        "prompt_id": "W3",
        "agents_involved": ["orchestrator", "synthesis_agent"],
        "tools_or_protocol": "A2A (transfer_to_agent)",
        "outcome": "ĐẠT",
        "notes": "transfer_to_agent('synthesis_agent') chạy đúng, nhận báo cáo executive đầy đủ về MCP và A2A.",
    },
    {
        "prompt_id": "W4",
        "agents_involved": ["orchestrator"],
        "tools_or_protocol": "suggest_routing",
        "outcome": "ĐẠT",
        "notes": "recommended_agent (điểm cao nhất) = synthesis_agent (0.365) do router bag-of-words, nhưng orchestrator tự lý luận và chọn đúng database_agent cho truy vấn SQL.",
    },
    {
        "prompt_id": "W5",
        "agents_involved": ["orchestrator"],
        "tools_or_protocol": "MCP sql_query — governance deny",
        "outcome": "ĐẠT",
        "notes": "Từ chối ngay tại orchestrator (không gọi sql_query), nêu rõ chỉ cho phép truy vấn đọc, không cho DROP TABLE.",
    },
]

print(f"{'ID':<4} {'Agents':<35} {'Protocol':<28} {'Kết quả':<12} Ghi chú")
print("-" * 100)
for row in adk_web_results:
    agents = ", ".join(row["agents_involved"])
    print(f"{row['prompt_id']:<4} {agents:<35} {row['tools_or_protocol']:<28} {row['outcome']:<12} {row['notes']}")

passed = sum(1 for r in adk_web_results if r["outcome"] == "ĐẠT")
print(f"\nTổng: {passed}/{len(adk_web_results)} prompt đạt yêu cầu")
print("\n💡 Sau khi điền xong, nộp notebook kèm screenshot ADK Web (ít nhất W1 và W2).")


---
## Module 5 — State, Bảo mật & Observability

### Quản lý State

| Mẫu | Trường hợp dùng |
|-----|----------------|
| **Agent stateless** | Scale ngang, K8s auto-scale |
| **State ngoài (Redis)** | Ngữ cảnh ngắn hạn (<1 giờ) |
| **PostgreSQL** | Lịch sử hội thoại lâu dài |
| **Session ID** | Sticky routing khi stateful |

**Thực hành tốt:** Thiết kế stateless trước; externalize state khi cần.

Trong ADK, dùng `tool_context.state` trong function tool để chia sẻ dữ liệu giữa các lượt.

In [37]:
# Ví dụ: state qua ToolContext (mẫu khái niệm)
example_tool = '''
from google.adk.tools.tool_context import ToolContext

async def track_cost(amount: float, tool_context: ToolContext) -> str:
    """Cộng dồn chi phí API vào session state."""
    total = tool_context.state.get("total_cost", 0.0) + amount
    tool_context.state["total_cost"] = total
    if total > 10.0:
        return f"Đã đạt trần chi phí (${total:.2f}). Cần phê duyệt của người."
    return f"Đã cập nhật chi phí: ${total:.2f}"
'''
print(example_tool)


from google.adk.tools.tool_context import ToolContext

async def track_cost(amount: float, tool_context: ToolContext) -> str:
    """Cộng dồn chi phí API vào session state."""
    total = tool_context.state.get("total_cost", 0.0) + amount
    tool_context.state["total_cost"] = total
    if total > 10.0:
        return f"Đã đạt trần chi phí (${total:.2f}). Cần phê duyệt của người."
    return f"Đã cập nhật chi phí: ${total:.2f}"



### Bảo mật & Governance

| Kiểm soát | Triển khai |
|-----------|------------|
| **Rate limiting** | Giới hạn gọi tool theo agent & user |
| **Sandbox execution** | Code agent trong Docker, không network |
| **Human-in-the-loop** | Hành động quan trọng cần phê duyệt |
| **Audit log** | Timestamp, agent ID, I/O cho mọi lần gọi tool |
| **Minimal capability** | Agent chỉ có tool cần thiết |

**Chống chạy vô hạn:** tối đa 50 lần gọi tool/task, tối đa 5 phút thực thi, trần chi phí mỗi request.

### Ma trận Capability Agent (từ slide)

| Agent | Được phép | Bị chặn |
|-------|-----------|--------|
| Research | search, đọc | ghi |
| Database | chỉ SELECT | DDL, DML |
| Email | soạn thảo | gửi (cần phê duyệt) |
| Code | chạy trong sandbox | deploy production |

### Data Governance — MCP & A2A (thực hành)

Hệ thống governance nằm trong `lab_utils/governance/`:

| Thành phần | Vai trò |
|------------|--------|
| `policy.json` | Ma trận capability: ai được gọi MCP/A2A tool nào |
| `GovernanceGuard` | Kiểm tra trước mỗi lần gọi (SQL read-only, rate limit, PII) |
| `AuditLogger` | Ghi audit vào `logs/governance_audit.jsonl` |
| `adk_callbacks.py` | `before_tool_callback` chặn tool vi phạm policy |

**Luồng kiểm soát:**

```
Request → GovernanceGuard → [ALLOW | DENY | HITL_REQUIRED]
                ↓
         AuditLogger (timestamp, actor, I/O)
                ↓
         MCP tool / A2A dispatch
```

In [38]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from lab_utils.governance import get_guard, AuditLogger

guard = get_guard()
audit = AuditLogger()

print("=== Ma trận MCP (orchestrator) ===")
conn = guard.authorize_mcp_connection("orchestrator")
print(f"Kết nối MCP: {conn.verdict.value} — tools: {conn.metadata.get('allowed_tools')}")

print("\n=== Kiểm tra vi phạm SQL (MCP) ===")
bad_sql = guard.authorize_mcp_tool(
    "orchestrator", "sql_query", {"sql": "DROP TABLE agent_metrics"}
)
print(f"DROP TABLE → {bad_sql.verdict.value}: {bad_sql.reason}")

good_sql = guard.authorize_mcp_tool(
    "orchestrator", "sql_query", {"sql": "SELECT * FROM agent_metrics"}
)
print(f"SELECT hợp lệ → {good_sql.verdict.value}")

print("\n=== Kiểm tra A2A dispatch ===")
allowed = guard.authorize_a2a_dispatch(
    "orchestrator", "search_agent", trace_id="demo-trace-001"
)
print(f"orchestrator → search_agent: {allowed.verdict.value}")

blocked = guard.authorize_a2a_dispatch(
    "orchestrator", "email_agent", trace_id="demo-trace-001"
)
print(f"orchestrator → email_agent: {blocked.verdict.value}: {blocked.reason}")

no_trace = guard.authorize_a2a_dispatch("orchestrator", "database_agent")
print(f"Không có trace_id → {no_trace.verdict.value}: {no_trace.reason}")

print()
print("=== Bai tap 5.2 -- synthesis_agent trong allowed_targets ===")
synth_allowed = guard.authorize_a2a_dispatch(
    "orchestrator", "synthesis_agent", trace_id="demo-trace-002"
)
print(f"orchestrator -> synthesis_agent: {synth_allowed.verdict.value}")

print()
print("=== Bai tap 5.2 -- chan tu khoa 'password' trong search_documents ===")
blocked_kw = guard.authorize_mcp_tool(
    "orchestrator", "search_documents", {"query": "tim password cua admin"}
)
print(f"query co 'password' -> {blocked_kw.verdict.value}: {blocked_kw.reason}")

ok_kw = guard.authorize_mcp_tool(
    "orchestrator", "search_documents", {"query": "kien truc MCP la gi"}
)
print(f"query binh thuong -> {ok_kw.verdict.value}")

print("\n=== PII → HITL ===")
pii_sql = guard.authorize_mcp_tool(
    "orchestrator",
    "sql_query",
    {"sql": "SELECT * FROM agent_metrics WHERE email = 'user@vinuni.edu.vn'"},
)
print(f"PII trong SQL → {pii_sql.verdict.value}: {pii_sql.reason}")

print("\n=== Tóm tắt audit log ===")
print(audit.summary())

=== Ma trận MCP (orchestrator) ===
Kết nối MCP: allow — tools: ['search_documents', 'sql_query', 'summarize_text', 'count_words']

=== Kiểm tra vi phạm SQL (MCP) ===
DROP TABLE → deny: Chỉ cho phép SELECT (read-only)
SELECT hợp lệ → allow

=== Kiểm tra A2A dispatch ===
orchestrator → search_agent: allow
orchestrator → email_agent: deny: 'orchestrator' không được dispatch A2A tới 'email_agent'. Chỉ cho phép: ['search_agent', 'database_agent', 'synthesis_agent']
Không có trace_id → hitl_required: A2A dispatch yêu cầu trace_id trong metadata (W3C Trace Context)

=== Bai tap 5.2 -- synthesis_agent trong allowed_targets ===
orchestrator -> synthesis_agent: allow

=== Bai tap 5.2 -- chan tu khoa 'password' trong search_documents ===
query co 'password' -> deny: Từ khóa 'password' bị chặn trong search_documents
query binh thuong -> allow

=== PII → HITL ===
PII trong SQL → hitl_required: Phát hiện dữ liệu nhạy cảm (PII) — cần phê duyệt HITL

=== Tóm tắt audit log ===
{'allow': 32, 'deny': 11,

### 📝 Bài tập 5.2 — Mở rộng chính sách governance

1. Mở `lab_utils/governance/policy.json` và thêm agent `synthesis_agent` vào `allowed_targets` của orchestrator.
2. Thêm rule chặn từ khóa `password` trong `search_documents`.
3. Chạy lại cell trên và xác nhận audit log ghi đủ sự kiện `deny` / `hitl_required`.
4. *(Nâng cao)* Viết test đảm bảo caller không hợp lệ không mở được kết nối MCP.

### Observability — Distributed Tracing

Một **trace ID** duy nhất nên bao trùm toàn bộ request multi-agent:

```
Trace ID: abc-123
└── Orchestrator (2.3s)
    ├── Search Agent (0.8s)
    │   └── web_search (0.5s)
    └── DB Agent (1.1s)
        └── sql_query (0.9s)
```

ADK truyền metadata qua ranh giới A2A qua `RunConfig.custom_metadata`.

In [39]:
import uuid

from google.adk.agents.run_config import RunConfig

trace_id = str(uuid.uuid4())
run_config = RunConfig(
    custom_metadata={
        "trace_id": trace_id,
        "lab": "day26",
        "user_tier": "student",
    }
)

print(f"Trace ID: {trace_id}")
print(f"Các khóa metadata: {list(run_config.custom_metadata.keys())}")
print("\nTrên A2A agent remote, metadata xuất hiện dạng:")
print('  event.custom_metadata["a2a_metadata"]["trace_id"]')

Trace ID: d35ae2ed-c1c8-4528-af55-4eaecc190863
Các khóa metadata: ['trace_id', 'lab', 'user_tier']

Trên A2A agent remote, metadata xuất hiện dạng:
  event.custom_metadata["a2a_metadata"]["trace_id"]


### Các metric cần theo dõi

| Metric | Mục đích |
|--------|----------|
| `tasks_completed` / `tasks_failed` | Độ tin cậy |
| `avg_task_duration` | Độ trễ |
| `tool_call_count` | Phát hiện chạy vô hạn |
| `cost_per_task` | Phân bổ ngân sách |
| `queue_depth` | Lập kế hoạch năng lực |

### 📝 Bài tập 5.1 — Thiết kế chính sách Governance

Với hệ 4-agent nghiên cứu, viết ma trận capability:
1. Tool nào mỗi agent được gọi
2. Hành động nào cần phê duyệt người
3. Rate limit theo agent
4. Số lần gọi tool và thời gian thực thi tối đa

---
## Lab #26 Capstone — Hạ tầng Multi-Agent

**Mục tiêu:** Xây hệ 4 agent (orchestrator + 3 specialist) qua A2A + MCP với trace đầy đủ và demo ADK Web.

**Thời gian:** 2 giờ

> ⛔ **Trước cell kiểm tra bên dưới:** A2A servers phải đang chạy (Module 0.5). Khuyến nghị capstone:
> ```bash
> bash scripts/start_capstone.sh
> ```
> hoặc `bash scripts/start_a2a_servers.sh` rồi `bash scripts/start_adk_web.sh`.

### Checklist

- [ ] MCP server với 3 tool (stdio spawn tự động qua McpToolset)
- [ ] Agent registry có health check
- [ ] Semantic router + `suggest_routing` tool trên orchestrator
- [ ] Search agent expose qua `to_a2a()` cổng 8011
- [ ] Database agent expose qua `to_a2a()` cổng 8012
- [ ] **Synthesis agent** expose qua `to_a2a()` cổng 8013
- [ ] Orchestrator tiêu thụ cả ba qua `RemoteA2aAgent`
- [ ] ADK Web demo — 5 prompt W1–W5 (Module 4) ghi kết quả
- [ ] Trace ID tự sinh trong ADK Web (auto-init governance)
- [ ] Chính sách governance ghi audit (`logs/governance_audit.jsonl`)

### Thử thách mở rộng

1. **Transport SSE:** Triển khai MCP server với FastAPI + uvicorn cổng 8080
2. **Tải đồng thời:** Gửi 5 request nghiên cứu và quan sát phân phối routing
3. **Embedding router:** Thay bag-of-words bằng `text-embedding-004`
4. **Cổng HITL:** Tạm dừng trước hành động vượt $10 chi phí API

In [40]:
# Script kiểm tra capstone — chạy khi A2A servers đã lên (Module 0.5)
import httpx

checks = {
    "search_agent_card": "http://localhost:8011/.well-known/agent-card.json",
    "database_agent_card": "http://localhost:8012/.well-known/agent-card.json",
    "synthesis_agent_card": "http://localhost:8013/.well-known/agent-card.json",
}

START_HINT = """
✗ A2A servers chưa chạy. Làm một trong các cách sau:

  Cách 1 (khuyến nghị capstone):
    bash scripts/start_capstone.sh

  Cách 2 (notebook): quay lại cell「Khởi động A2A servers nền」ở Module 0.5

  Cách 3 (terminal tại thư mục dự án):
    bash scripts/start_a2a_servers.sh

  Cách 4 (ba terminal):
    uvicorn agents.search_agent.agent:a2a_app --host localhost --port 8011
    uvicorn agents.database_agent.agent:a2a_app --host localhost --port 8012
    uvicorn agents.synthesis_agent.agent:a2a_app --host localhost --port 8013
"""

all_ok = True
for name, url in checks.items():
    try:
        r = httpx.get(url, timeout=5)
        ok = r.status_code == 200
        err = ""
    except Exception as exc:
        ok = False
        err = str(exc)
    status = "ĐẠT" if ok else "CHƯA ĐẠT"
    print(f"[{status}] {name}" + (f" — {err}" if err else ""))
    all_ok = all_ok and ok

if all_ok:
    print("\n✓ Sẵn sàng demo capstone")
    print("  Tiếp theo: bash scripts/start_capstone.sh → http://localhost:8020")
else:
    print(START_HINT)

[ĐẠT] search_agent_card
[ĐẠT] database_agent_card
[ĐẠT] synthesis_agent_card

✓ Sẵn sàng demo capstone
  Tiếp theo: bash scripts/start_capstone.sh → http://localhost:8020


---
## Tổng kết — Điểm then chốt

1. **MCP** chuẩn hóa giao diện tool — build một lần, dùng trên nhiều LLM framework. Triển khai MCP server trên K8s với mẫu registry.
2. **A2A** là microservices cho AI — hợp đồng, auth, observability. Google ADK expose/tiêu thụ agent qua `to_a2a()` và `RemoteA2aAgent`.
3. **Trí tuệ routing** ở lớp orchestrator tốt hơn chọn agent cứng — đầu tư semantic routing + fallback chain.

---

## Xem trước buổi sau

**Ngày 27:** Data Observability & Advanced Monitoring — Great Expectations, phát hiện bất thường, dbt tests, SLO engineering.

### Bài tập về nhà

- Hoàn thành checklist capstone Lab #26
- Đọc: [Tài liệu Great Expectations checkpoint](https://docs.greatexpectations.io/)
- Đọc: Google SRE Workbook — chương SLO engineering